In [ ]:
import os
import librosa
import numpy as np

def preprocess_and_process_audio(data_dir, target_length_seconds=1, sampling_rate=22050, n_mfcc=40):
    """
    Preprocesses and processes 1-second audio segments from two folders,
    extracting MFCC features.

    Args:
        data_dir (str): Path to the directory containing two class folders.
        target_length_seconds (int): Target length of each segment in seconds.
        sampling_rate (int): Desired sampling rate for the audio.
        n_mfcc (int): Number of MFCCs to extract.

    Returns:
        tuple: A tuple containing:
            - features (numpy.ndarray): Array of extracted MFCC features.
            - labels (numpy.ndarray): Array of corresponding numerical labels.
    """

    features =
    labels =
    class_labels = {}  # Mapping from class names to numerical labels
    label_count = 0

    # 1. Get class folders
    class_folders = [folder for folder in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, folder))]

    if len(class_folders) != 2:
        raise ValueError("The data directory must contain exactly two class folders.")

    for class_name in class_folders:
        class_dir = os.path.join(data_dir, class_name)
        class_labels[class_name] = label_count
        label_count += 1

        # 2. Iterate through audio files
        for filename in os.listdir(class_dir):
            file_path = os.path.join(class_dir, filename)
            if not file_path.lower().endswith(('.wav', '.mp3', '.flac')):
                continue

            try:
                # 3. Load audio and resample
                audio, sr = librosa.load(file_path, sr=sampling_rate)

                # 4. Ensure correct length
                target_length_samples = int(target_length_seconds * sampling_rate)

                if len(audio) > target_length_samples:
                    # Trim audio if it's longer
                    audio = audio[:target_length_samples]
                elif len(audio) < target_length_samples:
                    # Pad audio with zeros if it's shorter
                    padding = target_length_samples - len(audio)
                    audio = np.pad(audio, (0, padding), 'constant')

                # 5. Normalize amplitude
                audio_normalized = audio / np.max(np.abs(audio))

                # 6. Feature Extraction (MFCCs)
                mfccs = librosa.feature.mfcc(y=audio_normalized, sr=sampling_rate, n_mfcc=n_mfcc)
                mfccs_mean = np.mean(mfccs.T, axis=0)  # Average MFCCs over time

                # 7. Append to lists
                features.append(mfccs_mean)
                labels.append(class_labels[class_name])

            except Exception as e:
                print(f"Error processing file {filename}: {e}")

    return np.array(features), np.array(labels)

if __name__ == "__main__":
    data_dir = "audio_segments"  # Replace with your data directory

    # Create dummy directories and files if they don't exist (for testing)
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
        os.makedirs(os.path.join(data_dir, "class_1"))
        os.makedirs(os.path.join(data_dir, "class_2"))
        # Create dummy audio files (replace with your actual files)
        librosa.tone(200, sr=22050, length=1).astype(np.float32).tofile(os.path.join(data_dir, "class_1", "audio1.wav"))
        librosa.tone(500, sr=22050, length=1).astype(np.float32).tofile(os.path.join(data_dir, "class_2", "audio2.wav"))

    try:
        features, labels = preprocess_and_process_audio(data_dir)

        # Now 'features' contains a NumPy array of MFCC features,
        # and 'labels' contains the corresponding numerical labels.

        print(f"Shape of features: {features.shape}")
        print(f"Shape of labels: {labels.shape}")
        print(f"Example features: {features[0]}")
        print(f"Example label: {labels[0]}")

    except ValueError as e:
        print(f"Error: {e}")